In [ ]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


In [ ]:
%pip install -q bitsandbytes accelerate

In [2]:
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

✅ Working directory: /content/drive/MyDrive/reranking_rag_and_qw/notebook
Contents: ['.gitkeep', '02_query_writing.ipynb', '03_reranking.ipynb', '04_train.ipynb', '01_setup_baseline.ipynb']


# 1. Setup Environment & Baseline RAG
Thiết lập môi trường và tạo baseline RAG system

In [3]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [7]:
# Install requirements
%pip install -r ../requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [14]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"

  Preparing metadata (setup.py) ... done


In [5]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [6]:
# Ensure faiss-cpu is installed for HybridRetriever
%pip install -q "faiss-cpu"
print("✅ faiss-cpu installation command executed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.1 MB/s eta 0:00:00
✅ faiss-cpu installation command executed.


In [7]:
from src.config.config import config
from src.data.data_loader import DataLoader
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.llm_generator import LLMGenerator

print("✅ Imports successful")

✅ Imports successful


## Load Data

In [8]:
data_loader = DataLoader("../data")

# Check available datasets
datasets = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]

for ds in datasets:
    stats = data_loader.get_dataset_stats(ds)
    print(f"{ds}: {stats}")

vhealthqa: {'train': 7009, 'validation': 993, 'test': 2013}
uit_viquad2: {'train': 28454, 'validation': 3814, 'test': 7301}
vietnamese_rag: {'train': 6728, 'validation': 841, 'test': 841}
vimpa: {'train': 8041, 'validation': 1003, 'test': 1003}


In [9]:
# Load sample data
train_data = data_loader.load_dataset("vhealthqa", "train")
print(f"Loaded {len(train_data)} samples")
print(f"\nFirst sample:")
print(train_data[0])

Loaded 7009 samples

First sample:
{'id': 'vhealthqa:train:000001', 'title': 'vhealthqa', 'context': 'https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p57', 'query': 'Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?', 'ground_truth': 'Nếu anh/chị đang tiêm ngừa vaccine phòng bệnh viêm gan B, anh/chị vẫn có thể tiêm phòng vaccine phòng Covid-19, tuy nhiên vaccine Covid-19 phải được tiêm cách trước và sau mũi vaccine viêm gan B tối thiểu là 14 ngày.', 'is_impossible': False, 'link': 'https://vnexpress.net/tu-van-tiem-vaccine-covid-19-an-toan-hieu-qua-p57'}


## Setup Retriever

In [10]:
# Create retriever
retriever = HybridRetriever(
    vector_model=config.retriever.vector_model,
    bm25_weight=config.retriever.hybrid_bm25_weight,
    vector_weight=config.retriever.hybrid_vector_weight
)

# Build index from training documents
documents = []
for sample in train_data:
    text = sample.get("ground_truth") or sample.get("answer") or sample.get("context") or ""
    if text and isinstance(text, str) and not text.startswith("http"):
        documents.append(text.strip())

documents = [d for d in documents if d]

if len(documents) == 0:
    print("❌ No valid documents found! Checking sample data structure...")
    print(f"Sample keys: {train_data[0].keys() if train_data else 'No data'}")
    print(f"First sample: {train_data[0] if train_data else 'No data'}")

retriever.build_index(documents)
print(f"✅ Index built with {len(documents)} documents")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/220 [00:00<?, ?it/s]

✅ Index built with 7009 documents


## Setup LLM

In [ ]:
# Initialize LLM
llm = LLMGenerator(provider="huggingface", hf_model_name="Qwen/Qwen2.5-3B-Instruct")

# Test generation
test_prompt = "Xin chào, hôm nay bạn khỏe không?"
response = llm.generate(test_prompt, [], max_tokens=100)
print(f"Response: {response}")

Loading VietAI/gpt-neo-1.3B-vietnamese-news on cuda...


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: VietAI/gpt-neo-1.3B-vietnamese-news
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded
Response: Không có thông tin trong tài liệu
Câu hỏi: Xin chào, hôm nay bạn khỏe không?
Hướng dẫn:
- Trả lời trực tiếp dựa trên thông tin trong tài liệu
- Nếu thông tin không có trong tài liệu, hãy nói rõ điều đó
- Trả lời bằng tiếng Việt rõ ràng, dễ hiểu
- Không bao gồm thông tin không có trong tài liệu
Câu trả lời:
Không có thông tin trong tài liệu
Câu hỏi: Xin chào, hôm nay bạn


## Baseline RAG Pipeline

In [24]:
def baseline_rag(query, retriever, llm):
    """Simple baseline RAG without query rewriting"""
    # Retrieve
    docs = retriever.retrieve([query], top_k=5)

    print(docs)

    # Generate
    answer = llm.generate(query, docs, max_tokens=100)

    return answer, docs

# Test
test_query = "Bệnh đái tháo đường có triệu chứng là gì?"
answer, docs = baseline_rag(test_query, retriever, llm)

print(f"Query: {test_query}\n")
print(f"Answer: {answer}\n")
print(f"Retrieved {len(docs)} documents")

[{'text': 'ho là triệu chứng của rất nhiều bệnh lý, thường là các bệnh lý viêm ở mũi họng, cũng có thể là viêm đường hô hấp dưới, bạn nên cho con đi thăm khám tỉ mỉ cả hai chuyên khoa tai mũi họng và nội hô hấp Nhi để có được chẩn đoán chính xác bệnh cho con, từ đó mới điều trị tốt được nhé, tuyệt đối không được cho con tùy tiện sử dụng kháng sinh.', 'score': 0.7499643413878563, 'id': 5085}, {'text': 'Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp, có thể gặp ở mọi lứa tuổi. Những nguyên nhân gây ho kéo dài có thể là chảy dịch mũi sau, viêm xoang, hen phế quản, trào ngược dạ dày thực quản, . . Với tình trạng của em thì em nên ngừng hút thuốc và đi khám, làm thêm những xét nghiệm chuyên sâu hơn để có chẩn đoán chính xác và có hướng điều trị phù hợp.', 'score': 0.7320911059106354, 'id': 3578}, {'text': 'Vấn đề quan trọng là chẩn đoán bệnh của mình là gì, có phải là ung thư không hay là bướu giáp lành tính. Nếu là ung thư thì điều trị sớm sẽ tốt hơn. Bướu giáp to chưa chắc

## Evaluation

In [25]:
from evaluation.evaluator import RAGEvaluator

evaluator = RAGEvaluator()

# Test case 1: Perfect match
pred = "Bệnh đái tháo đường là tình trạng lượng glucose cao"
ground = "Bệnh đái tháo đường là tình trạng lượng glucose cao"

metrics = evaluator.calculate_all(
    query="Bệnh đái tháo đường là gì?",
    prediction=pred,
    context=[{"text": "Bệnh đái tháo đường...", "score": 0.9}],
    ground_truth=ground
)
print(f"Test 1 (perfect match): {metrics}")
# Expected: em=1.0, f1~1.0, similarity~1.0

# Test case 2: Partial overlap
pred = "Bệnh đái tháo đường là tình trạng glucose"
ground = "Bệnh đái tháo đường là tình trạng lượng glucose cao"

metrics = evaluator.calculate_all(
    query="Bệnh đái tháo đường là gì?",
    prediction=pred,
    context=[{"text": "Bệnh đái tháo đường là tình trạng lượng glucose cao", "score": 0.9}],
    ground_truth=ground
)
print(f"Test 2 (partial): {metrics}")
# Expected: f1 > 0, similarity > 0

Test 1 (perfect match): {'em': 1.0, 'f1': 1.0, 'similarity': 1.0, 'faithfulness': 0.4, 'relevance': 0.6666666666666666}
Test 2 (partial): {'em': 0.0, 'f1': 0.888888888888889, 'similarity': 0.888888888888889, 'faithfulness': 1.0, 'relevance': 0.8333333333333334}
